<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.1-fastmcp/practice/GCP_Capstone_7.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 7.1 — Building FastMCP Server

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup — install FastMCP, imports, and the DocuMind server

Run this first. It installs FastMCP, imports what every exercise below needs, and defines the shared in-memory document store plus the four DocuMind tools on a single `mcp` instance. Later exercises reuse these names, so the notebook runs top-to-bottom.

> No Vertex/Gemini client is needed for this lesson — 7.1 is pure MCP-server construction. Auth and `genai.Client(enterprise=True, ...)` come back in later Module 7 lessons when tools call the model.

In [ ]:
%%bash
pip install -q "fastmcp>=4,<5"

In [ ]:
import fastmcp
print(f'FastMCP version: {fastmcp.__version__}')

import asyncio, os
from fastmcp import FastMCP, Client
from fastmcp.exceptions import ToolError
from typing import Literal

# Shared DocuMind MCP server instance (reused by every exercise below)
mcp = FastMCP('DocuMind')

# Simulated document store
DOCUMENTS = [
    {'id': 'doc-001', 'title': 'Q4 Financial Report', 'category': 'financial',
     'content': 'Revenue increased 15% year-over-year...', 'pages': 24},
    {'id': 'doc-002', 'title': 'Engineering Design Spec', 'category': 'technical',
     'content': 'System architecture uses microservices...', 'pages': 18},
    {'id': 'doc-003', 'title': 'Employee Handbook 2026', 'category': 'hr',
     'content': 'Company policies and procedures...', 'pages': 45},
    {'id': 'doc-004', 'title': 'Marketing Strategy Q2', 'category': 'marketing',
     'content': 'Target audience analysis shows growth...', 'pages': 12},
]
print(f'Document store: {len(DOCUMENTS)} documents')


@mcp.tool
def search_documents(query: str, max_results: int = 5) -> list[dict]:
    """Search the document repository for relevant documents.

    Args:
        query: Search query to match against document titles and content.
        max_results: Maximum results to return (default 5, max 20).
    """
    if not query.strip():
        raise ToolError('Search query cannot be empty')
    results = []
    for doc in DOCUMENTS:
        if query.lower() in doc['title'].lower() or query.lower() in doc['content'].lower():
            results.append({'id': doc['id'], 'title': doc['title'],
                            'category': doc['category'], 'relevance': 0.95})
    return results[:min(max_results, 20)]


@mcp.tool
def calculate_cost(
    page_count: int,
    processing_type: Literal['standard', 'premium', 'enterprise'] = 'standard',
    include_ocr: bool = False
) -> dict:
    """Calculate processing cost for a document.

    Args:
        page_count: Number of pages (must be positive).
        processing_type: Tier — standard ($0.01), premium ($0.03), enterprise ($0.05).
        include_ocr: Add OCR processing at $0.02/page.
    """
    if page_count <= 0:
        raise ToolError('Page count must be positive')
    rates = {'standard': 0.01, 'premium': 0.03, 'enterprise': 0.05}
    ocr = 0.02 if include_ocr else 0.0
    total = round((rates[processing_type] + ocr) * page_count, 2)
    return {'page_count': page_count, 'processing_type': processing_type,
            'total_cost': total, 'currency': 'USD'}


@mcp.tool
def get_stats() -> dict:
    """Get summary statistics about the document repository.

    Returns total documents, pages, and category breakdown.
    """
    total_pages = sum(d['pages'] for d in DOCUMENTS)
    cats = {}
    for d in DOCUMENTS:
        cats[d['category']] = cats.get(d['category'], 0) + 1
    return {'total_documents': len(DOCUMENTS), 'total_pages': total_pages,
            'categories': cats, 'avg_pages': round(total_pages / len(DOCUMENTS), 1)}


@mcp.tool
def classify_document(title: str, content: str) -> dict:
    """Classify a document into a category based on title and content.

    Args:
        title: The document title.
        content: The document content text.
    """
    if not title.strip() or not content.strip():
        raise ToolError('Both title and content are required')
    text = (title + ' ' + content).lower()
    kws = {'financial': ['revenue', 'budget', 'profit'],
           'technical': ['api', 'system', 'architecture'],
           'hr': ['employee', 'policy', 'handbook'],
           'marketing': ['campaign', 'audience', 'brand']}
    scores = {c: sum(1 for k in ws if k in text) for c, ws in kws.items()}
    best = max(scores, key=scores.get)
    return {'category': best, 'confidence': round(min(scores[best] / 3, 1.0), 2)}

print('Registered 4 tools on the DocuMind server.')

## Exercise 1: First @mcp.tool

**Difficulty:** Easy

Create a server with one tool (get_stats). Run locally. Verify with fastmcp version.

1. pip install fastmcp
2. from fastmcp import FastMCP; mcp = FastMCP("test")
3. @mcp.tool a simple function
4. Verify fastmcp version output

In [ ]:
# Minimal one-tool server on its own instance, to see @mcp.tool in isolation.
import fastmcp
from fastmcp import FastMCP

print(f'FastMCP version: {fastmcp.__version__}')

demo = FastMCP('test')

def get_stats_demo() -> dict:
    """Return simple repository stats."""
    return {'total_documents': len(DOCUMENTS),
            'total_pages': sum(d['pages'] for d in DOCUMENTS)}

demo.tool(get_stats_demo)  # register with the MCP server (name stays a normal callable)

# The plain function still runs locally; the server exposes it over MCP
print('Local call:', get_stats_demo())

## Exercise 2: MCP Inspector Connect

**Difficulty:** Easy

Start server. Launch Inspector. Connect to http://localhost:8000/mcp. List tools.

1. python documind_server.py
2. npx @modelcontextprotocol/inspector
3. Select Streamable HTTP, enter URL
4. Verify 4 tools listed

In [ ]:
# Write the standalone server file that serves the 4 tools over Streamable HTTP.
server_code = '''
import asyncio, os, logging
from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from typing import Literal

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mcp = FastMCP("DocuMind")

DOCUMENTS = [
    {"id": "doc-001", "title": "Q4 Financial Report", "category": "financial",
     "content": "Revenue increased 15% year-over-year...", "pages": 24},
    {"id": "doc-002", "title": "Engineering Design Spec", "category": "technical",
     "content": "System architecture uses microservices...", "pages": 18},
    {"id": "doc-003", "title": "Employee Handbook 2026", "category": "hr",
     "content": "Company policies and procedures...", "pages": 45},
    {"id": "doc-004", "title": "Marketing Strategy Q2", "category": "marketing",
     "content": "Target audience analysis shows growth...", "pages": 12},
]

@mcp.tool
def search_documents(query: str, max_results: int = 5) -> list[dict]:
    """Search the document repository for relevant documents.
    Args:
        query: Search query to match against documents.
        max_results: Max results (default 5, max 20).
    """
    if not query.strip(): raise ToolError("Query cannot be empty")
    results = []
    for doc in DOCUMENTS:
        if query.lower() in doc["title"].lower() or query.lower() in doc["content"].lower():
            results.append({"id": doc["id"], "title": doc["title"],
                            "category": doc["category"], "relevance": 0.95})
    return results[:min(max_results, 20)]

@mcp.tool
def calculate_cost(page_count: int,
                   processing_type: Literal["standard","premium","enterprise"] = "standard",
                   include_ocr: bool = False) -> dict:
    """Calculate processing cost for a document.
    Args:
        page_count: Number of pages (positive).
        processing_type: standard ($0.01), premium ($0.03), enterprise ($0.05).
        include_ocr: Add OCR at $0.02/page.
    """
    if page_count <= 0: raise ToolError("Page count must be positive")
    rates = {"standard": 0.01, "premium": 0.03, "enterprise": 0.05}
    ocr = 0.02 if include_ocr else 0.0
    return {"total_cost": round((rates[processing_type] + ocr) * page_count, 2), "currency": "USD"}

@mcp.tool
def get_stats() -> dict:
    """Get repository statistics: total docs, pages, categories."""
    total_pages = sum(d["pages"] for d in DOCUMENTS)
    cats = {}
    for d in DOCUMENTS: cats[d["category"]] = cats.get(d["category"], 0) + 1
    return {"total_documents": len(DOCUMENTS), "total_pages": total_pages, "categories": cats}

@mcp.tool
def classify_document(title: str, content: str) -> dict:
    """Classify a document into a category.
    Args:
        title: Document title.
        content: Document content text.
    """
    if not title.strip() or not content.strip(): raise ToolError("Both title and content required")
    text = (title + " " + content).lower()
    kws = {"financial":["revenue","budget"],"technical":["api","system"],
           "hr":["employee","policy"],"marketing":["campaign","audience"]}
    scores = {c: sum(1 for k in ws if k in text) for c,ws in kws.items()}
    best = max(scores, key=scores.get)
    return {"category": best, "confidence": round(min(scores[best]/2, 1.0), 2)}

if __name__ == "__main__":
    port = int(os.getenv("PORT", 8000))
    logger.info(f"DocuMind MCP on port {port}")
    asyncio.run(mcp.run_async(transport="streamable-http", host="0.0.0.0", port=port))
'''

with open('documind_server.py', 'w') as f:
    f.write(server_code)
print('Wrote documind_server.py')
print(f'Size: {os.path.getsize("documind_server.py")} bytes')

In [ ]:
%%bash
# Run these in a LOCAL terminal (the Inspector is a browser UI, not available inside Colab).
# Terminal 1 — start the server (serves Streamable HTTP at http://localhost:8000/mcp):
#   python documind_server.py
#
# Terminal 2 — launch the MCP Inspector:
#   npx @modelcontextprotocol/inspector
#
# In the Inspector UI:
#   1. Transport Type: Streamable HTTP
#   2. URL: http://localhost:8000/mcp
#   3. Click Connect, open the Tools tab -> you should see 4 tools listed.
echo 'Start the server, then: npx @modelcontextprotocol/inspector'

## Exercise 3: Call from Inspector

**Difficulty:** Easy

Call search_documents from Inspector with query="financial". View structured response.

1. Click search_documents in Tools tab
2. Enter query: "financial"
3. Click Call, view response

In [ ]:
# In-notebook equivalent of the Inspector 'Call' button: connect the FastMCP
# Client directly to the in-memory server object and invoke search_documents.
async def call_search():
    async with Client(mcp) as client:
        result = await client.call_tool('search_documents', {'query': 'financial'})
        print('search_documents(query="financial") ->')
        print(result)

await call_search()

## Exercise 4: 4-Tool Server

**Difficulty:** Medium

Build all 4 DocuMind tools. Verify all appear in tools/list. Call each one.

1. search_documents, calculate_cost, get_stats, classify_document
2. Each with proper type hints and docstrings
3. Test all 4 from Inspector or Client

In [ ]:
# The 4 tools were registered on `mcp` in Setup. Confirm tools/list shows all 4,
# then call each one through the FastMCP Client.
async def verify_four_tools():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print(f'=== {len(tools)} Tools Found ===')
        for t in tools:
            print(f'  {t.name}: {t.description[:60]}...')

        print('\n=== Tool Calls ===')
        print('search  :', await client.call_tool('search_documents', {'query': 'financial'}))
        print('cost    :', await client.call_tool('calculate_cost',
                                                   {'page_count': 100, 'processing_type': 'enterprise'}))
        print('stats   :', await client.call_tool('get_stats', {}))
        print('classify:', await client.call_tool('classify_document',
                                                   {'title': 'Budget Plan', 'content': 'Revenue projections'}))

await verify_four_tools()

## Exercise 5: ToolError Validation

**Difficulty:** Medium

Send empty query, negative page_count, missing content. Verify ToolError messages.

1. search_documents(query="") → "cannot be empty"
2. calculate_cost(page_count=-5) → "must be positive"
3. classify_document(title="x", content="") → "required"

In [ ]:
# Each invalid call should surface the ToolError message back to the client.
async def test_errors():
    async with Client(mcp) as client:
        try:
            await client.call_tool('search_documents', {'query': ''})
        except Exception as e:
            print(f'Empty query error: {e}')

        try:
            await client.call_tool('calculate_cost', {'page_count': -5})
        except Exception as e:
            print(f'Negative pages error: {e}')

        try:
            await client.call_tool('classify_document', {'title': 'Test', 'content': ''})
        except Exception as e:
            print(f'Missing content error: {e}')

await test_errors()

## Exercise 6: curl Testing

**Difficulty:** Medium

Initialize, list tools, call calculate_cost via curl JSON-RPC.

1. POST initialize to /mcp
2. POST tools/list
3. POST tools/call with calculate_cost args

In [ ]:
%%bash
# Run against a LIVE server started with `python documind_server.py`
# (Streamable HTTP endpoint at http://localhost:8000/mcp).
# Streamable HTTP requires the Accept header to allow BOTH json and event-stream.
ACCEPT='Accept: application/json, text/event-stream'
CT='Content-Type: application/json'
URL='http://localhost:8000/mcp'

# 1. initialize the session
curl -s -X POST "$URL" -H "$CT" -H "$ACCEPT" -d '{
  "jsonrpc": "2.0", "id": 1, "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "curl-test", "version": "1.0"}
  }
}'
echo '\n---'

# 2. list the tools
curl -s -X POST "$URL" -H "$CT" -H "$ACCEPT" -d '{
  "jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}
}'
echo '\n---'

# 3. call calculate_cost
curl -s -X POST "$URL" -H "$CT" -H "$ACCEPT" -d '{
  "jsonrpc": "2.0", "id": 3, "method": "tools/call",
  "params": {
    "name": "calculate_cost",
    "arguments": {"page_count": 100, "processing_type": "premium", "include_ocr": true}
  }
}'

## Exercise 7: FastMCP Client

**Difficulty:** Challenge

Write Python test script using fastmcp.Client. Test all 4 tools programmatically.

1. async with Client(url) as client
2. list_tools(), call_tool() for each
3. Assert expected results

In [ ]:
# Programmatic test harness with assertions on every tool.
# Client(mcp) connects in-memory; swap for Client('http://localhost:8000/mcp')
# to hit the running HTTP server instead.
async def test_all_tools():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        names = {t.name for t in tools}
        assert names == {'search_documents', 'calculate_cost', 'get_stats', 'classify_document'}, names
        print(f'PASS list_tools -> {sorted(names)}')

        r1 = await client.call_tool('search_documents', {'query': 'financial'})
        assert any(d['title'] == 'Q4 Financial Report' for d in r1.data), r1.data
        print('PASS search_documents finds Q4 Financial Report')

        r2 = await client.call_tool('calculate_cost',
                                    {'page_count': 100, 'processing_type': 'enterprise'})
        assert r2.data['total_cost'] == 5.0, r2.data
        print('PASS calculate_cost enterprise 100pp -> $5.00')

        r3 = await client.call_tool('get_stats', {})
        assert r3.data['total_documents'] == 4, r3.data
        print('PASS get_stats -> 4 documents')

        r4 = await client.call_tool('classify_document',
                                    {'title': 'Budget Plan', 'content': 'Revenue projections'})
        assert r4.data['category'] == 'financial', r4.data
        print('PASS classify_document -> financial')

        print('\nAll 4 tools verified.')

await test_all_tools()

## Exercise 8: Add a 5th Tool

**Difficulty:** Challenge

Add delete_document(doc_id) with ToolError for non-existent docs and confirmation requirement.

1. @mcp.tool with doc_id: str parameter
2. Check if doc_id exists in DOCUMENTS
3. Return confirmation message (do not actually delete)

In [ ]:
# Register a 5th tool on the same server. It validates the id and returns a
# confirmation request instead of destructively deleting (never hard-delete data).
@mcp.tool
def delete_document(doc_id: str) -> dict:
    """Request deletion of a document by id (requires confirmation; does not delete).

    Args:
        doc_id: The document id to delete, e.g. 'doc-001'.
    """
    if not doc_id.strip():
        raise ToolError('doc_id is required')
    match = next((d for d in DOCUMENTS if d['id'] == doc_id), None)
    if match is None:
        raise ToolError(f"Document '{doc_id}' not found")
    return {'action': 'delete_requested',
            'doc_id': doc_id,
            'title': match['title'],
            'requires_confirmation': True,
            'message': f"Confirm deletion of '{match['title']}' ({doc_id})?"}


async def test_delete():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print(f'Tools now: {len(tools)} ({sorted(t.name for t in tools)})')

        # Valid id -> confirmation payload
        ok = await client.call_tool('delete_document', {'doc_id': 'doc-002'})
        print('valid  :', ok.data)

        # Non-existent id -> ToolError
        try:
            await client.call_tool('delete_document', {'doc_id': 'doc-999'})
        except Exception as e:
            print('missing:', e)

await test_delete()